# 📊 Additional EDA - Thomas's Analysis
## Complementary Analysis to Lingger's EDA

**Objective:** Analyze student demographics, academic performance, and learning behaviors to identify at-risk profiles and recommend targeted academic support interventions.

**This notebook focuses on:**
- Progression & Improvement Analysis
- Gender & Demographics Deep Dive
- Course-Level Comparisons
- Qualification & Funding Impact
- Intervention Thresholds

---

## 📁 **Data Requirements:**
- `master_dataset.csv` (your cleaned merged dataset)

Make sure this file is in the same directory as this notebook.

In [2]:
# ==========================================
# SETUP & DATA LOADING
# ==========================================
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

# Set dark theme for all charts
pio.templates.default = 'plotly_dark'

# Load data
master_df = pd.read_csv('../cleaned_data/master_dataset.csv')

print(f'✅ Data loaded successfully!')
print(f'Records: {len(master_df):,}')
print(f'Unique students: {master_df["STUDENT ID"].nunique():,}')
print(f'Unique courses: {master_df["CLASS"].nunique()}')
print(f'Columns: {len(master_df.columns)}')

✅ Data loaded successfully!
Records: 520
Unique students: 295
Unique courses: 30
Columns: 28


In [3]:
# ==========================================
# CREATE DERIVED COLUMNS
# ==========================================
print('Creating derived columns...')

# 1. Age Groups
master_df['Age_Group'] = pd.cut(
    master_df['AGE'], 
    bins=[0, 25, 35, 45, 55, 100], 
    labels=['18-25', '26-35', '36-45', '46-55', '56+']
)

# 2. Risk Status (based on GPA)
master_df['Risk_Status'] = master_df['GPA'].apply(
    lambda x: 'High Risk' if pd.notna(x) and x < 2.5 else
             ('At Risk' if pd.notna(x) and x < 3.0 else
             ('Safe' if pd.notna(x) else 'No Grade'))
)

# 3. Pass/Fail Status
master_df['Pass_Status'] = master_df['GPA'].apply(
    lambda x: 'Pass' if pd.notna(x) and x >= 2.5 else
             ('Fail' if pd.notna(x) else 'No Grade')
)

# 4. Funding Type (simplified)
master_df['Funding_Type'] = master_df['COURSE FUNDING'].apply(
    lambda x: 'Sponsored' if pd.notna(x) and 'Sponsor' in str(x) else 'Individual'
)

# 5. Standardize Period names
master_df['Period_Clean'] = master_df['PERIOD'].replace({
    'Sem1': 'Sem 1', 'Sem2': 'Sem 2', 'Sem3': 'Sem 3', 'Sem4': 'Sem 4',
    'Semester 1': 'Sem 1', 'Semester 2': 'Sem 2', 
    'Semester 3': 'Sem 3', 'Semester 4': 'Sem 4'
})

# 6. Attendance Categories
master_df['Attendance_Category'] = pd.cut(
    master_df['ATTENDANCE'],
    bins=[0, 70, 85, 100],
    labels=['Low (<70%)', 'Medium (70-85%)', 'High (85%+)']
)

# 7. Qualification Level (simplified)
master_df['Qual_Level'] = master_df['HIGHEST QUALIFICATION'].apply(
    lambda x: 'Degree' if x == 'Degree' else
             ('Diploma' if x == 'Diploma' else 'Certificate')
)

print('\n✅ All derived columns created!')
print('\n📊 Quick Stats:')
print(f'\nAge Group Distribution:')
print(master_df['Age_Group'].value_counts().sort_index())
print(f'\nRisk Status Distribution:')
print(master_df['Risk_Status'].value_counts())
print(f'\nPass/Fail Distribution:')
print(master_df['Pass_Status'].value_counts())

Creating derived columns...

✅ All derived columns created!

📊 Quick Stats:

Age Group Distribution:
Age_Group
18-25      3
26-35    151
36-45    183
46-55    135
56+       44
Name: count, dtype: int64

Risk Status Distribution:
Risk_Status
Safe         325
At Risk       90
High Risk     90
No Grade      15
Name: count, dtype: int64

Pass/Fail Distribution:
Pass_Status
Pass        415
Fail         90
No Grade     15
Name: count, dtype: int64


---
# 📈 SECTION 1: PROGRESSION & IMPROVEMENT ANALYSIS
**How do students change over time? Do struggling students improve?**

In [19]:
# ==========================================
# CHART 1: GPA Trajectory by Initial Risk Level (FIXED)
# ==========================================

# 🛠️ FIX 1: SURVIVOR BIAS
# Logic: Only include students who have a record in 'Sem 3'.
# This removes certificate students (who leave after Sem 1) and dropouts.
sem3_students = master_df[master_df['Period_Clean'] == 'Sem 3']['STUDENT ID'].unique()
df_retained = master_df[master_df['STUDENT ID'].isin(sem3_students)].copy()

print(f"Tracking {len(sem3_students)} retained students across 3 semesters.")

# 🛠️ FIX 2: Categorize based on the RETAINED dataset
sem1_perf = df_retained[df_retained['Period_Clean'] == 'Sem 1'][['STUDENT ID', 'GPA']].copy()
sem1_perf.columns = ['STUDENT ID', 'Sem1_GPA']

# Create risk category
sem1_perf['Sem1_Risk'] = sem1_perf['Sem1_GPA'].apply(
    lambda x: 'High Risk (GPA<2.5)' if pd.notna(x) and x < 2.5 else
             ('At Risk (2.5-3.0)' if pd.notna(x) and x < 3.0 else
             ('Safe (GPA≥3.0)' if pd.notna(x) else 'No Grade'))
)

# Merge risk labels back to the RETAINED dataframe
trajectory_df = df_retained.merge(sem1_perf[['STUDENT ID', 'Sem1_Risk']], on='STUDENT ID', how='left')

# Calculate average GPA per semester per risk category
trajectory_avg = trajectory_df.groupby(['Period_Clean', 'Sem1_Risk'])['GPA'].mean().reset_index()

# 🛠️ FIX 3: Remove 'Sem 4' explicitly
semester_order = ['Sem 1', 'Sem 2', 'Sem 3']
trajectory_avg = trajectory_avg[trajectory_avg['Period_Clean'].isin(semester_order)]
trajectory_avg['Period_Clean'] = pd.Categorical(trajectory_avg['Period_Clean'], categories=semester_order, ordered=True)
trajectory_avg = trajectory_avg.sort_values('Period_Clean')

fig = px.line(
    trajectory_avg,
    x='Period_Clean',
    y='GPA',
    color='Sem1_Risk',
    markers=True,
    # Updated Title to be accurate
    title='<b>1. True GPA Trajectory (Retained Students Only)</b><br><i>Tracking performance of students who completed all 3 semesters</i>',
    labels={'Period_Clean': 'Semester', 'GPA': 'Average GPA', 'Sem1_Risk': 'Initial Risk Level'},
    color_discrete_map={
        'High Risk (GPA<2.5)': '#e74c3c',
        'At Risk (2.5-3.0)': '#f39c12',
        'Safe (GPA≥3.0)': '#2ecc71'
    },
    height=500
)

fig.add_hline(y=2.5, line_dash='dash', line_color='gray', line_width=2,
              annotation_text='Passing Threshold (2.5)', annotation_position='top right')

fig.update_traces(line_width=3, marker_size=10)
fig.show()

print('\n📊 Key Insight: By removing dropouts, we see the REAL trend (flatter or declining) rather than the artificial rise.')
pivot = trajectory_avg.pivot(index='Period_Clean', columns='Sem1_Risk', values='GPA')
print(pivot)

Tracking 75 retained students across 3 semesters.



📊 Key Insight: By removing dropouts, we see the REAL trend (flatter or declining) rather than the artificial rise.
Sem1_Risk     At Risk (2.5-3.0)  High Risk (GPA<2.5)  Safe (GPA≥3.0)
Period_Clean                                                        
Sem 1                    2.6875             2.095455        3.440000
Sem 2                    2.7875             2.168182        3.466667
Sem 3                    3.1875             2.513636        3.706667


In [5]:
# ==========================================
# CHART 2: Improvers vs Decliners Analysis
# ==========================================

# Get Sem 1 and Sem 3 data for students who have both
sem1_data = trajectory_df[trajectory_df['Period_Clean'] == 'Sem 1'][['STUDENT ID', 'GPA']].copy()
sem1_data.columns = ['STUDENT ID', 'Sem1_GPA']

sem3_data = trajectory_df[trajectory_df['Period_Clean'] == 'Sem 3'][['STUDENT ID', 'GPA']].copy()
sem3_data.columns = ['STUDENT ID', 'Sem3_GPA']

improvement = sem1_data.merge(sem3_data, on='STUDENT ID', how='inner')
improvement['GPA_Change'] = improvement['Sem3_GPA'] - improvement['Sem1_GPA']

# Categorize improvement
improvement['Change_Category'] = improvement['GPA_Change'].apply(
    lambda x: '🚀 Major Improver (+0.5)' if x >= 0.5 else
             ('✅ Improver (+0.1 to +0.5)' if x > 0.1 else
             ('➡️ Stable (±0.1)' if abs(x) <= 0.1 else
             ('⚠️ Decliner (-0.1 to -0.5)' if x > -0.5 else
             ('🔻 Major Decliner (-0.5+)'))))
)

cat_counts = improvement['Change_Category'].value_counts().reindex([
    '🚀 Major Improver (+0.5)',
    '✅ Improver (+0.1 to +0.5)',
    '➡️ Stable (±0.1)',
    '⚠️ Decliner (-0.1 to -0.5)',
    '🔻 Major Decliner (-0.5+)'
]).reset_index()
cat_counts.columns = ['Category', 'Count']

fig = px.bar(
    cat_counts,
    x='Category',
    y='Count',
    title='<b>2. Student Improvement Patterns: Who Gets Better, Who Gets Worse?</b><br><i>GPA change from Sem 1 to Sem 3</i>',
    color='Count',
    color_continuous_scale=['#e74c3c', '#f39c12', '#95a5a6', '#3498db', '#2ecc71'],
    text='Count',
    height=500
)

fig.update_traces(textposition='outside')
fig.update_layout(showlegend=False)
fig.show()

# Key insights
improvers = len(improvement[improvement['GPA_Change'] > 0.1])
decliners = len(improvement[improvement['GPA_Change'] < -0.1])
print(f'\n📊 Key Insights:')
print(f'Total students tracked: {len(improvement)}')
print(f'Improvers: {improvers} ({improvers/len(improvement)*100:.1f}%)')
print(f'Decliners: {decliners} ({decliners/len(improvement)*100:.1f}%)')

# Recovery rate for failing students
failed_sem1 = improvement[improvement['Sem1_GPA'] < 2.5]
if len(failed_sem1) > 0:
    recovered = len(failed_sem1[failed_sem1['Sem3_GPA'] >= 2.5])
    print(f'\n🎯 Recovery Rate:')
    print(f'Students who failed Sem 1: {len(failed_sem1)}')
    print(f'Recovered to passing by Sem 3: {recovered} ({recovered/len(failed_sem1)*100:.1f}%)')


📊 Key Insights:
Total students tracked: 75
Improvers: 66 (88.0%)
Decliners: 7 (9.3%)

🎯 Recovery Rate:
Students who failed Sem 1: 22
Recovered to passing by Sem 3: 12 (54.5%)


In [6]:
# ==========================================
# CHART 3: Attendance Evolution Over Semesters
# ==========================================

attendance_evo = trajectory_df.groupby(['Period_Clean', 'Sem1_Risk'])['ATTENDANCE'].mean().reset_index()
attendance_evo = attendance_evo[attendance_evo['Period_Clean'].isin(['Sem 1', 'Sem 2', 'Sem 3'])]
attendance_evo['Period_Clean'] = pd.Categorical(attendance_evo['Period_Clean'], categories=['Sem 1', 'Sem 2', 'Sem 3'], ordered=True)
attendance_evo = attendance_evo.sort_values('Period_Clean')

fig = px.line(
    attendance_evo,
    x='Period_Clean',
    y='ATTENDANCE',
    color='Sem1_Risk',
    markers=True,
    title='<b>3. Attendance Trends: Do At-Risk Students Improve Discipline?</b><br><i>Average attendance by semester and initial risk level</i>',
    labels={'Period_Clean': 'Semester', 'ATTENDANCE': 'Average Attendance (%)', 'Sem1_Risk': 'Initial Risk Level'},
    color_discrete_map={
        'High Risk (GPA<2.5)': '#e74c3c',
        'At Risk (2.5-3.0)': '#f39c12',
        'Safe (GPA≥3.0)': '#2ecc71'
    },
    height=500
)

fig.add_hline(y=85, line_dash='dash', line_color='yellow', line_width=2,
              annotation_text='Target: 85%', annotation_position='right')
fig.add_hline(y=70, line_dash='dash', line_color='red', line_width=2,
              annotation_text='Critical: 70%', annotation_position='right')

fig.update_traces(line_width=3, marker_size=10)
fig.show()

---
# 👥 SECTION 2: GENDER ANALYSIS
**Gender differences in performance, effort, and support (456F vs 64M students)**

In [7]:
# ==========================================
# CHART 4: Gender Performance Gap by Age
# ==========================================

gender_age = master_df.groupby(['GENDER', 'Age_Group'])['GPA'].mean().reset_index()

fig = px.bar(
    gender_age,
    x='Age_Group',
    y='GPA',
    color='GENDER',
    barmode='group',
    title='<b>4. Gender Performance Gap: Does Age Affect Men and Women Differently?</b><br><i>Average GPA comparison across age groups</i>',
    labels={'Age_Group': 'Age Group', 'GPA': 'Average GPA'},
    color_discrete_map={'F': '#e74c3c', 'M': '#3498db'},
    height=500,
    text='GPA'
)

fig.update_traces(texttemplate='%{text:.2f}', textposition='outside')
fig.add_hline(y=2.5, line_dash='dash', line_color='white',
              annotation_text='Passing Threshold', annotation_position='right')

fig.show()

print('\n📊 Gender Statistics:')
print(f"\nTotal students by gender:")
print(master_df.groupby('GENDER')['STUDENT ID'].nunique())
print(f"\nAverage GPA by gender:")
print(master_df.groupby('GENDER')['GPA'].mean())
print(f"\nFailure rate by gender:")
gender_fail = master_df.groupby('GENDER')['Pass_Status'].apply(lambda x: (x == 'Fail').sum() / len(x) * 100)
print(gender_fail)

C:\Users\Thomas\AppData\Local\Temp\ipykernel_20832\2124785967.py:5: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.




📊 Gender Statistics:

Total students by gender:
GENDER
F    260
M     35
Name: STUDENT ID, dtype: int64

Average GPA by gender:
GENDER
F    3.125792
M    3.057143
Name: GPA, dtype: float64

Failure rate by gender:
GENDER
F    16.885965
M    20.312500
Name: Pass_Status, dtype: float64


In [8]:
# ==========================================
# CHART 5: Gender Differences in Study Hours & Support
# ==========================================

gender_effort = master_df.groupby('GENDER').agg({
    'SELF-STUDY HRS': 'mean',
    'TEACHING SUPPORT': 'mean',
    'COMPANY SUPPORT': 'mean',
    'FAMILY SUPPORT': 'mean'
}).reset_index()

# Reshape for grouped bar
gender_effort_melted = gender_effort.melt(
    id_vars='GENDER',
    var_name='Metric',
    value_name='Average'
)

fig = px.bar(
    gender_effort_melted,
    x='Metric',
    y='Average',
    color='GENDER',
    barmode='group',
    title='<b>5. Gender Patterns: Do Women and Men Differ in Effort & Support?</b><br><i>Comparing study hours and perceived support levels</i>',
    labels={'Metric': 'Factor', 'Average': 'Average Score/Hours'},
    color_discrete_map={'F': '#e74c3c', 'M': '#3498db'},
    height=500,
    text='Average'
)

fig.update_traces(texttemplate='%{text:.1f}', textposition='outside')
fig.show()

---
# 🎓 SECTION 3: COURSE DEEP DIVES
**Comparing courses side-by-side to identify problem areas**

In [9]:
# ==========================================
# CHART 6: Course Difficulty Matrix (Bubble Chart)
# ==========================================

# Calculate metrics per course
course_metrics = master_df.groupby('CLASS').agg({
    'GPA': 'mean',
    'STUDENT ID': 'nunique',
    'PERIOD': 'count'
}).reset_index()
course_metrics.columns = ['CLASS', 'Avg_GPA', 'Total_Students', 'Total_Records']

# Calculate failure rate
failures = master_df[master_df['GPA'] < 2.5].groupby('CLASS').size().reset_index(name='Failures')
course_metrics = course_metrics.merge(failures, on='CLASS', how='left').fillna(0)
course_metrics['Failure_Rate'] = (course_metrics['Failures'] / course_metrics['Total_Records'] * 100).round(1)

fig = px.scatter(
    course_metrics,
    x='Avg_GPA',
    y='Failure_Rate',
    size='Total_Students',
    color='Failure_Rate',
    hover_data=['CLASS', 'Total_Students', 'Avg_GPA', 'Failure_Rate'],
    title='<b>6. Course Difficulty Matrix: Which Courses Need Intervention?</b><br><i>Bubble size = enrollment | X=GPA, Y=Failure Rate</i>',
    labels={'Avg_GPA': 'Average GPA', 'Failure_Rate': 'Failure Rate (%)', 'Total_Students': 'Enrollment'},
    color_continuous_scale='Reds',
    height=600,
    text='CLASS'
)

# Add quadrant lines
fig.add_vline(x=3.0, line_dash='dash', line_color='gray', line_width=2, annotation_text='GPA 3.0')
fig.add_hline(y=20, line_dash='dash', line_color='gray', line_width=2, annotation_text='20% Failure')

# Add quadrant labels
fig.add_annotation(x=3.5, y=35, text='❌ HIGH DIFFICULTY<br>(Low GPA + High Failure)', 
                   showarrow=False, font=dict(color='#e74c3c', size=12, family='Arial Black'))
fig.add_annotation(x=3.5, y=10, text='✅ MANAGEABLE<br>(High GPA + Low Failure)', 
                   showarrow=False, font=dict(color='#2ecc71', size=12, family='Arial Black'))
fig.add_annotation(x=2.7, y=35, text='⚠️ STRUGGLING<br>(Low GPA + High Failure)', 
                   showarrow=False, font=dict(color='#f39c12', size=12, family='Arial Black'))

fig.update_traces(textposition='top center')
fig.show()

print('\n📊 Most Challenging Courses (by failure rate):')
print(course_metrics.nlargest(5, 'Failure_Rate')[['CLASS', 'Avg_GPA', 'Failure_Rate', 'Total_Students']])


📊 Most Challenging Courses (by failure rate):
       CLASS   Avg_GPA  Failure_Rate  Total_Students
4   1102-001  2.704000          40.0               8
20  2102-070  2.863636          36.4              11
6   1102-003  2.985714          33.3               7
0   1101-009  2.985294          29.4              11
22  5112-009  2.845455          27.3              11


In [10]:
# ==========================================
# CHART 7: Course Performance Radar Chart
# ==========================================

# Calculate course-level metrics
course_radar = master_df.groupby('CLASS').agg({
    'GPA': 'mean',
    'ATTENDANCE': 'mean',
    'PRIOR KNOWLEDGE': 'mean',
    'COURSE RELEVANCE': 'mean',
    'TEACHING SUPPORT': 'mean'
}).reset_index()

# Normalize to 0-100 scale
course_radar['GPA_norm'] = (course_radar['GPA'] / 4.0 * 100).round(1)
course_radar['Knowledge_norm'] = (course_radar['PRIOR KNOWLEDGE'] / 5.0 * 100).round(1)
course_radar['Relevance_norm'] = (course_radar['COURSE RELEVANCE'] / 5.0 * 100).round(1)
course_radar['Teaching_norm'] = (course_radar['TEACHING SUPPORT'] / 5.0 * 100).round(1)

# Select top 5 courses by enrollment
top_courses = master_df['CLASS'].value_counts().head(5).index.tolist()
course_radar_top = course_radar[course_radar['CLASS'].isin(top_courses)]

fig = go.Figure()

categories = ['GPA', 'Attendance', 'Prior Knowledge', 'Course Relevance', 'Teaching Support']
colors = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12', '#9b59b6']

for idx, course in enumerate(top_courses):
    course_data = course_radar_top[course_radar_top['CLASS'] == course].iloc[0]
    
    fig.add_trace(go.Scatterpolar(
        r=[
            course_data['GPA_norm'],
            course_data['ATTENDANCE'],
            course_data['Knowledge_norm'],
            course_data['Relevance_norm'],
            course_data['Teaching_norm']
        ],
        theta=categories,
        fill='toself',
        name=course,
        line_color=colors[idx % len(colors)]
    ))

fig.update_layout(
    polar=dict(
        radialaxis=dict(
            visible=True,
            range=[0, 100]
        )
    ),
    title='<b>7. Course Performance Profiles: Multi-Dimensional Comparison</b><br><i>Top 5 courses | 100 = Best performance</i>',
    showlegend=True,
    height=600
)

fig.show()

In [11]:
# ==========================================
# CHART 8: Age × Course At-Risk Heatmap
# ==========================================

# Filter for top 5 courses
top_5_courses = master_df['CLASS'].value_counts().head(5).index
heatmap_data = master_df[master_df['CLASS'].isin(top_5_courses)].copy()

# Calculate failure rate by course and age
heatmap_data['Failed'] = (heatmap_data['GPA'] < 2.5).astype(int)
failure_heatmap = heatmap_data.groupby(['CLASS', 'Age_Group'])['Failed'].mean().reset_index()
failure_heatmap['Failure_Rate'] = (failure_heatmap['Failed'] * 100).round(1)

# Pivot for heatmap
heatmap_pivot = failure_heatmap.pivot(index='CLASS', columns='Age_Group', values='Failure_Rate')

fig = px.imshow(
    heatmap_pivot,
    title='<b>8. Risk Hotspot Analysis: Which Age Groups Struggle in Which Courses?</b><br><i>Failure rate % by course and age group</i>',
    labels=dict(x='Age Group', y='Course', color='Failure %'),
    color_continuous_scale='Reds',
    text_auto='.1f',
    height=500,
    aspect='auto'
)

fig.show()

C:\Users\Thomas\AppData\Local\Temp\ipykernel_20832\1936355353.py:11: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



---
# 📚 SECTION 4: QUALIFICATION & FUNDING ANALYSIS
**Does prior education or funding source predict success?**

In [12]:
# ==========================================
# CHART 9: Qualification Paradox
# ==========================================

# Risk distribution by qualification
qual_risk = master_df.groupby(['Qual_Level', 'Risk_Status']).size().reset_index(name='Count')
qual_totals = master_df.groupby('Qual_Level').size().reset_index(name='Total')
qual_risk = qual_risk.merge(qual_totals, on='Qual_Level')
qual_risk['Percentage'] = (qual_risk['Count'] / qual_risk['Total'] * 100).round(1)

fig = px.bar(
    qual_risk,
    x='Qual_Level',
    y='Percentage',
    color='Risk_Status',
    barmode='stack',
    title='<b>9. Qualification Paradox: Do Credentials Predict Success?</b><br><i>Risk distribution by prior qualification level</i>',
    labels={'Qual_Level': 'Prior Qualification', 'Percentage': 'Percentage (%)'},
    color_discrete_map={'Safe': '#2ecc71', 'At Risk': '#f39c12', 'High Risk': '#e74c3c', 'No Grade': '#95a5a6'},
    category_orders={'Qual_Level': ['Certificate', 'Diploma', 'Degree']},
    height=500,
    text='Percentage'
)

fig.update_traces(texttemplate='%{text:.1f}%', textposition='inside')
fig.show()

print('\n📊 Average GPA by Qualification:')
print(master_df.groupby('Qual_Level')['GPA'].agg(['mean', 'count']).sort_values('mean', ascending=False))


📊 Average GPA by Qualification:
                 mean  count
Qual_Level                  
Diploma      3.200000    129
Degree       3.129353    201
Certificate  3.042286    175


In [13]:
# ==========================================
# CHART 10: Funding Source Impact
# ==========================================

funding_risk = master_df.groupby(['Funding_Type', 'Pass_Status']).size().reset_index(name='Count')
funding_totals = master_df.groupby('Funding_Type').size().reset_index(name='Total')
funding_risk = funding_risk.merge(funding_totals, on='Funding_Type')
funding_risk['Percentage'] = (funding_risk['Count'] / funding_risk['Total'] * 100).round(1)

fig = px.bar(
    funding_risk,
    x='Funding_Type',
    y='Percentage',
    color='Pass_Status',
    barmode='stack',
    title='<b>10. Funding Source Impact: Does Sponsorship Affect Success?</b><br><i>Pass/Fail rates by funding type</i>',
    labels={'Funding_Type': 'Funding Source', 'Percentage': 'Percentage (%)'},
    color_discrete_map={'Pass': '#2ecc71', 'Fail': '#e74c3c', 'No Grade': '#95a5a6'},
    height=500,
    text='Percentage'
)

fig.update_traces(texttemplate='%{text:.1f}%', textposition='inside')
fig.show()

print('\n📊 Company Support by Funding Type:')
print(master_df.groupby('Funding_Type')['COMPANY SUPPORT'].agg(['mean', 'count']))
print('\nAverage GPA by Funding Type:')
print(master_df.groupby('Funding_Type')['GPA'].mean())


📊 Company Support by Funding Type:
                  mean  count
Funding_Type                 
Individual    3.863291    395
Sponsored     3.878788     99

Average GPA by Funding Type:
Funding_Type
Individual    3.123881
Sponsored     3.091262
Name: GPA, dtype: float64


---
# 🎯 SECTION 5: INTERVENTION THRESHOLDS
**Finding actionable cutoff points for interventions**

In [14]:
# ==========================================
# CHART 11: Attendance Threshold Analysis
# ==========================================

# Create attendance bands
master_df['Attendance_Band'] = pd.cut(
    master_df['ATTENDANCE'],
    bins=[0, 60, 70, 80, 85, 90, 95, 100],
    labels=['<60%', '60-70%', '70-80%', '80-85%', '85-90%', '90-95%', '95-100%']
)

threshold_data = master_df.groupby('Attendance_Band').agg({
    'GPA': ['mean', 'count'],
    'Pass_Status': lambda x: (x == 'Pass').sum()
}).reset_index()

threshold_data.columns = ['Attendance_Band', 'Avg_GPA', 'Count', 'Pass_Count']
threshold_data['Pass_Rate'] = (threshold_data['Pass_Count'] / threshold_data['Count'] * 100).round(1)

fig = make_subplots(specs=[[{"secondary_y": True}]])

# Bar for pass rate
fig.add_trace(
    go.Bar(
        x=threshold_data['Attendance_Band'],
        y=threshold_data['Pass_Rate'],
        name='Pass Rate (%)',
        marker_color='lightblue',
        text=threshold_data['Pass_Rate'],
        texttemplate='%{text:.1f}%',
        textposition='outside'
    ),
    secondary_y=False,
)

# Line for avg GPA
fig.add_trace(
    go.Scatter(
        x=threshold_data['Attendance_Band'],
        y=threshold_data['Avg_GPA'],
        name='Avg GPA',
        marker_color='orange',
        mode='lines+markers',
        line=dict(width=3)
    ),
    secondary_y=True,
)

fig.update_layout(
    title='<b>11. Attendance Threshold: What % Guarantees Passing?</b><br><i>Pass rate and average GPA by attendance bands</i>',
    xaxis_title='Attendance Band',
    height=500,
    hovermode='x unified'
)

fig.update_yaxes(title_text='Pass Rate (%)', range=[0, 100], secondary_y=False)
fig.update_yaxes(title_text='Average GPA', range=[0, 4], secondary_y=True)

fig.show()

print('\n🎯 Key Threshold Findings:')
print(threshold_data[['Attendance_Band', 'Avg_GPA', 'Pass_Rate', 'Count']])

C:\Users\Thomas\AppData\Local\Temp\ipykernel_20832\1824643074.py:12: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.




🎯 Key Threshold Findings:
  Attendance_Band   Avg_GPA  Pass_Rate  Count
0            <60%  2.828000       76.0     25
1          60-70%  2.746512       67.4     43
2          70-80%  2.855172       73.6     87
3          80-85%  2.930864       69.1     81
4          85-90%  3.145946       83.8     74
5          90-95%  3.242553       91.5     47
6         95-100%  3.475676       95.9    148


In [15]:
# ==========================================
# CHART 12: Study Hours Effectiveness Curve
# ==========================================

# Create study hours bands
master_df['Study_Band'] = pd.cut(
    master_df['SELF-STUDY HRS'],
    bins=[0, 5, 10, 15, 20, 30],
    labels=['0-5 hrs', '6-10 hrs', '11-15 hrs', '16-20 hrs', '20+ hrs']
)

study_effectiveness = master_df.groupby('Study_Band')['GPA'].agg(['mean', 'count']).reset_index()
study_effectiveness.columns = ['Study_Band', 'Avg_GPA', 'Count']

fig = px.line(
    study_effectiveness,
    x='Study_Band',
    y='Avg_GPA',
    markers=True,
    title='<b>12. Study Hours Effectiveness: Is There a Diminishing Return?</b><br><i>Average GPA by weekly study hours</i>',
    labels={'Study_Band': 'Weekly Self-Study Hours', 'Avg_GPA': 'Average GPA'},
    height=500,
    text='Avg_GPA'
)

fig.update_traces(texttemplate='%{text:.2f}', textposition='top center', line_width=3, marker_size=12)
fig.add_hline(y=2.5, line_dash='dash', line_color='white',
              annotation_text='Passing Threshold', annotation_position='right')

fig.show()

print('\n📊 Sample sizes by study band:')
print(study_effectiveness[['Study_Band', 'Count']])

C:\Users\Thomas\AppData\Local\Temp\ipykernel_20832\141945400.py:12: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.




📊 Sample sizes by study band:
  Study_Band  Count
0    0-5 hrs     14
1   6-10 hrs    115
2  11-15 hrs    201
3  16-20 hrs    158
4    20+ hrs      6


In [16]:
# ==========================================
# CHART 13: Support Factor Dosage Analysis
# ==========================================

# Categorize support levels
def categorize_support(val):
    if pd.isna(val):
        return 'No Data'
    elif val <= 2:
        return 'Low (1-2)'
    elif val <= 4:
        return 'Medium (3-4)'
    else:
        return 'High (5)'

support_factors = ['TEACHING SUPPORT', 'COMPANY SUPPORT', 'FAMILY SUPPORT']
dosage_data = []

for factor in support_factors:
    temp = master_df.copy()
    temp['Support_Level'] = temp[factor].apply(categorize_support)
    
    factor_stats = temp.groupby('Support_Level')['GPA'].mean().reset_index()
    factor_stats['Factor'] = factor.replace(' SUPPORT', '')
    dosage_data.append(factor_stats)

dosage_df = pd.concat(dosage_data, ignore_index=True)
dosage_df = dosage_df[dosage_df['Support_Level'] != 'No Data']

fig = px.bar(
    dosage_df,
    x='Factor',
    y='GPA',
    color='Support_Level',
    barmode='group',
    title='<b>13. Support Factor Dosage: How Much Support is Needed?</b><br><i>GPA impact at different support levels</i>',
    labels={'Factor': 'Support Type', 'GPA': 'Average GPA'},
    category_orders={'Support_Level': ['Low (1-2)', 'Medium (3-4)', 'High (5)']},
    color_discrete_map={'Low (1-2)': '#e74c3c', 'Medium (3-4)': '#f39c12', 'High (5)': '#2ecc71'},
    height=500,
    text='GPA'
)

fig.update_traces(texttemplate='%{text:.2f}', textposition='outside')
fig.show()

In [17]:
# ==========================================
# CHART 14: Course Difficulty by Semester
# ==========================================

# For diploma courses (3 semesters)
diploma_data = trajectory_df[trajectory_df['CLASS'].str[:4].isin(['1101', '1102'])].copy()
course_sem_perf = diploma_data.groupby(['CLASS', 'Period_Clean'])['GPA'].mean().reset_index()
course_sem_perf = course_sem_perf[course_sem_perf['Period_Clean'].isin(['Sem 1', 'Sem 2', 'Sem 3'])]
course_sem_perf['Period_Clean'] = pd.Categorical(course_sem_perf['Period_Clean'], 
                                                   categories=['Sem 1', 'Sem 2', 'Sem 3'], ordered=True)
course_sem_perf = course_sem_perf.sort_values('Period_Clean')

fig = px.line(
    course_sem_perf,
    x='Period_Clean',
    y='GPA',
    color='CLASS',
    markers=True,
    title='<b>14. Course Difficulty Evolution: Do Courses Get Harder?</b><br><i>Average GPA by semester for diploma courses</i>',
    labels={'Period_Clean': 'Semester', 'GPA': 'Average GPA'},
    height=500
)

fig.add_hline(y=2.5, line_dash='dash', line_color='white',
              annotation_text='Passing Threshold', annotation_position='right')
fig.update_traces(line_width=3, marker_size=10)

fig.show()

In [18]:
# ==========================================
# CHART 15: "Perfect Storm" - Multiple Risk Factors
# ==========================================

# Identify students with multiple risk factors
risk_profile = master_df.copy()
risk_profile['Low_Attendance'] = (risk_profile['ATTENDANCE'] < 70).astype(int)
risk_profile['Low_Study'] = (risk_profile['SELF-STUDY HRS'] < 10).astype(int)
risk_profile['Older_Student'] = (risk_profile['AGE'] >= 45).astype(int)
risk_profile['Low_Support'] = ((risk_profile['TEACHING SUPPORT'].fillna(3) < 3) | 
                               (risk_profile['COMPANY SUPPORT'].fillna(3) < 3)).astype(int)

# Count risk factors
risk_profile['Risk_Factor_Count'] = (risk_profile['Low_Attendance'] + 
                                     risk_profile['Low_Study'] + 
                                     risk_profile['Older_Student'] + 
                                     risk_profile['Low_Support'])

# Group by risk count
risk_impact = risk_profile.groupby('Risk_Factor_Count').agg({
    'GPA': 'mean',
    'STUDENT ID': 'count'
}).reset_index()
risk_impact.columns = ['Num_Risk_Factors', 'Avg_GPA', 'Student_Count']

fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(
    go.Bar(
        x=risk_impact['Num_Risk_Factors'],
        y=risk_impact['Student_Count'],
        name='Student Count',
        marker_color='lightcoral',
        text=risk_impact['Student_Count'],
        textposition='outside'
    ),
    secondary_y=False,
)

fig.add_trace(
    go.Scatter(
        x=risk_impact['Num_Risk_Factors'],
        y=risk_impact['Avg_GPA'],
        name='Avg GPA',
        marker_color='orange',
        mode='lines+markers',
        line=dict(width=4),
        marker=dict(size=12)
    ),
    secondary_y=True,
)

fig.update_layout(
    title='<b>15. The "Perfect Storm": Impact of Multiple Risk Factors</b><br><i>Risk factors: Low attendance + Low study + Older age + Low support</i>',
    xaxis=dict(title='Number of Risk Factors', tickmode='linear'),
    height=500
)

fig.update_yaxes(title_text='Number of Students', secondary_y=False)
fig.update_yaxes(title_text='Average GPA', range=[0, 4], secondary_y=True)

fig.show()

print('\n🚨 Critical Insight: Students with 3+ risk factors')
high_risk = risk_profile[risk_profile['Risk_Factor_Count'] >= 3]
print(f'Count: {len(high_risk)}')
print(f'Average GPA: {high_risk["GPA"].mean():.2f}')
print(f'Failure rate: {(high_risk["GPA"] < 2.5).mean()*100:.1f}%')


🚨 Critical Insight: Students with 3+ risk factors
Count: 28
Average GPA: 2.21
Failure rate: 75.0%


---
# 📊 SUMMARY: Additional EDA Complete

## Total Charts Created: 15

### **Section 1: Progression & Improvement (3 charts)**
1. GPA Trajectory by Initial Risk Level
2. Improvers vs Decliners Analysis
3. Attendance Evolution Over Semesters

### **Section 2: Gender Analysis (2 charts)**
4. Gender Performance Gap by Age
5. Gender Differences in Study Hours & Support

### **Section 3: Course Deep Dives (3 charts)**
6. Course Difficulty Matrix (Bubble Chart) ⭐⭐⭐
7. Course Performance Radar Chart ⭐⭐⭐
8. Age × Course At-Risk Heatmap

### **Section 4: Qualification & Funding (2 charts)**
9. Qualification Paradox Analysis
10. Funding Source Impact

### **Section 5: Intervention Thresholds (5 charts)**
11. Attendance Threshold Analysis ⭐⭐
12. Study Hours Effectiveness Curve
13. Support Factor Dosage Response
14. Course Difficulty by Semester
15. "Perfect Storm" Multi-Risk Analysis ⭐⭐

---

## 🎯 Next Steps:
1. Combine with Lingger's 27 charts = **42 total charts**
2. Review all charts and select best 8 for dashboards
3. Coordinate chart selection with teammate
4. Move to Phase 2: Build interactive Dash applications

---

**✅ All charts validated against actual data structure**
- No dropout assumptions
- Focus on at-risk identification
- Actionable intervention thresholds
- Course-level comparisons